# Yellow Taxi Data Analysis for Whole 2023 Year

In [1]:
# Try use the data in 20023
import polars as pl
import glob
import os
import gc
import glob

## Step 1: Load Lazy Mode Through Polars Scan

In [10]:
data = pl.read_parquet(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2023\yellow_taxi\yellow_tripdata_2023-01.parquet")
data.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
i64,datetime[ns],datetime[ns],f64,f64,f64,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,"""N""",161,141,2,9.3,1.0,0.5,0.0,0.0,1.0,14.3,2.5,0.0
2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.1,1.0,"""N""",43,237,1,7.9,1.0,0.5,4.0,0.0,1.0,16.9,2.5,0.0
2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,"""N""",48,238,1,14.9,1.0,0.5,15.0,0.0,1.0,34.9,2.5,0.0
1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.9,1.0,"""N""",138,7,1,12.1,7.25,0.5,0.0,0.0,1.0,20.85,0.0,1.25
2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,"""N""",107,79,1,11.4,1.0,0.5,3.28,0.0,1.0,19.68,2.5,0.0


## Step 2: Features Engineering

In [1]:
import polars as pl
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import glob

# Initialize model (logistic regression using SGD)
clf = SGDClassifier(loss="log_loss")
classes = [0,1]  # adjust to your Payment_Type labels

# For storing true labels and predicted probabilities (for ROC/metrics)
y_true_all = []
y_pred_all = []
y_score_all = []

batch_size = 100_000
files = glob.glob(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2023\yellow_taxi\yellow_tripdata_2023-*.parquet")

for f in files:
    # Build lazy frame (data manipulation happens here)
    lf = (
        pl.scan_parquet(f)
        .select(["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "payment_type",
                        "fare_amount", "tip_amount"])
               .filter((pl.col("passenger_count") >= 0) & (pl.col("trip_distance") >= 0) & (pl.col("trip_distance") <= 50) & 
                       (pl.col("fare_amount") >= 0) & (pl.col("tip_amount") >= 0))
               .filter(
                       pl.col("payment_type") == 1)
               .with_columns(
                       ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("duration_days"))
               .filter(
                       (pl.col("duration_days") == 0))
               .with_columns(
                       ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("duration_seconds"))
               .with_columns(
                       pl.when(pl.col("tip_amount") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("tip_category"))
               .select(["passenger_count", "trip_distance", "fare_amount", "duration_seconds", "tip_category"])
    )

    # Stream through this file in batches
    start = 0
    while True:
        # Collect only a tiny slice to keep memory low
        batch = lf.slice(start, batch_size).collect(engine="streaming")
        if batch.height == 0:
            break  # no more rows

        # Prepare features/labels
        X = batch.drop("tip_category").to_numpy()
        y = batch["tip_category"].to_numpy()

        # Train incrementally
        clf.partial_fit(X, y, classes=classes)

        # Store predictions for metrics
        y_pred = clf.predict(X)
        y_score = clf.predict_proba(X)  # needed for ROC

        y_true_all.extend(y)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

        start += batch_size

# --- Calculate metrics ---
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_score_all = np.array(y_score_all)

accuracy = accuracy_score(y_true_all, y_pred_all)
precision = precision_score(y_true_all, y_pred_all, average='macro')
recall = recall_score(y_true_all, y_pred_all, average='macro')
f1 = f1_score(y_true_all, y_pred_all, average='macro')

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)


Accuracy : 0.9248492536832926
Precision: 0.5153325904685561
Recall   : 0.5106781540137719
F1 Score : 0.5118444521917882


In [3]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# y_true should be shape (n_samples,), with labels 0/1
# y_score should be predicted probabilities for class 1 (positive class)

fpr, tpr, thresholds = roc_curve(y_true_all, y_score_all)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

ValueError: y should be a 1d array, got an array of shape (21898545, 2) instead.

In [2]:

# --- ROC curve (macro average for multi-class) ---
# sklearn's roc_curve works for one-vs-rest binary targets,
# so we need to binarize labels for multi-class ROC.
from sklearn.preprocessing import label_binarize
n_classes = len(classes)
y_true_bin = label_binarize(y_true_all, classes=classes)

# Calculate ROC curve and AUC for each class
fpr, tpr, roc_auc = dict(), dict(), dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score_all[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(8,6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {classes[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


IndexError: index 1 is out of bounds for axis 1 with size 1

In [ ]:
''' data manipulation
yellow_2009 = (
    yellow_2009.select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
)
''' 